# Wan2.2 video LatentLPIPS featurizer training

This notebook is the experiment driver for a video adaptation of [Diffusion2GAN / E-LatentLPIPS](https://arxiv.org/html/2405.05967v2) in the Wan2.2 TI2V-5B latent space. The reusable data, caching, model, checkpoint, evaluation, and training implementation lives in the importable wan_lpips.featurizer package module.

The paper trains VGG16-BN on images encoded by the target VAE, removes early pooling, and calibrates five normalized intermediate feature distances with nonnegative 1x1 heads. Here:

- Wan's frozen VAE maps RGB video in [-1,1] to an already-standardized 48-channel latent with stride (4,16,16). The 5B DiT and T5 are not loaded.
- The 2D VGG is applied to each causal latent frame and the calibrated maps are averaged over latent time and space. Four early pools are removed to match Wan's 16x spatial VAE stride.
- Scalar pair similarities are globally mapped to dissimilarities and fitted with Huber regression. This assumes interval-scale scores and is an adaptation of BAPPS's reference/two-candidate ranking protocol.
- With a Wan-latent classification-pretrained VGG, set train_trunk=False and provide trunk_checkpoint for paper-style frozen-trunk calibration. The default joint ImageNet-initialized trunk tuning is an explicitly labeled fallback.
- There is no output bias or latent L1 term, so identical inputs have distance zero.


## Dataset contract

Use JSONL or CSV with video_a, video_b, and score. Optional columns are split, group_id, latent_a, and latent_b. Paths are relative to video_root or the manifest directory. Explicit cached latents must contain a latents or latent safetensors tensor plus Wan provenance metadata.

All raw videos are deterministically sampled to 4k+1 frames, resized with an aspect-preserving center crop, normalized to [-1,1], and cached before DataLoader workers start. Content aliases and group IDs are checked to prevent train/validation leakage.


In [ ]:
import importlib.util

from wan_lpips.featurizer import (
    FeaturizerConfig,
    PACKAGE_ROOT,
    plot_training_diagnostics,
    resolve_device,
    run_contract_tests,
    run_training_pipeline,
    validate_config,
)


In [ ]:
cfg = FeaturizerConfig(
    repo_root=PACKAGE_ROOT,
    manifest_path=PACKAGE_ROOT / 'wan-lpips' / 'pairs.jsonl',
    cache_dir=PACKAGE_ROOT / 'wan-lpips' / 'latent_cache',
    output_dir=PACKAGE_ROOT / 'wan-lpips' / 'outputs',
    clip_frames=17,
    resize_hw=(256, 256),
    score_kind='similarity',
    score_min=0.0,
    score_max=1.0,
    remove_first_pools=4,
    initialize_from_imagenet=True,
    train_trunk=True,
    batch_size=4,
    num_workers=4,
    epochs=10,
    fixed_lr_epochs=5,
    head_lr=1e-4,
    trunk_lr=1e-5,
    amp_dtype='bfloat16',
    seed=1234,
)

validate_config(cfg)
print(f'repo:       {cfg.repo_root}')
print(f'manifest:   {cfg.manifest_path}')
print(f'checkpoint: {cfg.checkpoint_dir}')
print(f'device:     {resolve_device(cfg)}')
if importlib.util.find_spec('av') is None:
    print('Install PyAV before caching raw videos: pip install av')
if importlib.util.find_spec('einops') is None:
    print('Install einops before loading the local Wan VAE: pip install einops')


In [ ]:
run_contract_tests()
print('Synthetic shape, identity, symmetry, nonnegativity, augmentation, and label tests passed.')


In [ ]:
training_state = {}
if not cfg.manifest_path.is_file():
    print(f'Add the pair manifest at {cfg.manifest_path}, then rerun this cell.')
else:
    training_state = run_training_pipeline(cfg)
    print(f"pairs: {len(training_state['records'])}")
    print(f"cache: {training_state['cache_stats']}")
    print(f"best validation metrics: {training_state['metrics']}")


In [ ]:
plot_training_diagnostics(training_state)


## Deployment

For a downstream loss, reconstruct FeaturizerConfig from the saved checkpoint settings, call wan_lpips.featurizer.build_metric, load payload['metric'], switch to eval mode, and freeze parameters. The model forward itself does not use no_grad, so gradients still flow to generated latent inputs.
